In [ ]:
#Stage 1: verify workflows + label style evidence (YAML + follow-called-files)

In [1]:
# ============================================================
# Step 1: Verify GitHub Actions workflows from URL list (V16-aligned)
# - Input: URL list CSV(s) in a folder (expects column: full_name)
# - Output: verified_workflows_v16.csv (only repos with >= 1 GHA workflow)
# - For each workflow: fetch YAML + optional called-file following + V16-aligned detection
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm  # pip install tqdm
except ImportError:
    tqdm = None


# ============================================================
# CONFIG
# ============================================================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

# Folder containing your URL list CSV(s)
URL_LIST_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext"
)

# Which CSVs in that folder to read
URL_LIST_GLOB = "*.csv"  # e.g., "URL_List*.csv" if you want to restrict

OUT_DIR = URL_LIST_DIR  # you can change this
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_VERIFIED_WORKFLOWS_CSV = OUT_DIR / "verified_workflows_v16.csv"

DEFAULT_BRANCH_ONLY = True  # use default branch ref for contents
FOLLOW_CALLED_FILES_GH = True
MAX_FOLLOW_DEPTH_GH = 2
MAX_FOLLOW_BYTES_GH = 1_500_000

# ----------- API robustness knobs -----------
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000


# ============================================================
# Helpers
# ============================================================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def dot_fullname_to_slash(s: str) -> str:
    s = (s or "").strip()
    if not s:
        return ""
    s = s.replace("https://github.com/", "").replace("http://github.com/", "")
    s = s.split("?")[0].strip().strip("/")
    if "/" in s:
        return s
    if "." in s:
        owner, repo = s.split(".", 1)
        return f"{owner}/{repo}"
    return s

def b64_to_text(content_b64: str) -> Optional[str]:
    try:
        return base64.b64decode(content_b64).decode("utf-8", errors="ignore")
    except Exception:
        return None

def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    if csv_path.exists():
        return
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    """
    Limit tokens to GITHUB_TOKEN_1..N (by file order).
    """
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def load_repo_fullnames_from_url_list_folder(folder: Path, glob_pat: str) -> List[str]:
    """
    Reads all CSV files in `folder` that match `glob_pat`.
    Primary expected column: full_name
    Also tolerates: url, repo_url, github_url
    Returns unique repos (owner/repo) in original order.
    """
    if not folder.exists():
        raise FileNotFoundError(f"URL list folder not found: {folder}")

    csv_files = sorted(folder.glob(glob_pat))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {folder} matching {glob_pat}")

    repos: List[str] = []
    for p in csv_files:
        try:
            with p.open("r", encoding="utf-8", errors="ignore", newline="") as f:
                rdr = csv.DictReader(f)
                cols = [c.lower().strip() for c in (rdr.fieldnames or [])]
                # choose best column
                col = None
                for candidate in ("full_name", "repo", "repository", "url", "repo_url", "github_url"):
                    if candidate in cols:
                        col = candidate
                        break
                if not col:
                    continue

                # map back to original header case
                header_map = {c.lower().strip(): c for c in (rdr.fieldnames or [])}
                real_col = header_map[col]

                for row in rdr:
                    v = (row.get(real_col) or "").strip()
                    if not v:
                        continue
                    repo = dot_fullname_to_slash(v)
                    if repo and "/" in repo:
                        repos.append(repo)
        except Exception:
            continue

    return unique_preserve(repos)


# ============================================================
# GitHub API client (token rotation + retries)
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "workflow-verifier-v16/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException as e:
                print(f"[net] {method} {url} attempt {attempt}: {e}")
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                # keep going; treat as None
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# ============================================================
# GitHub endpoints
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    return list(gh.paginate(url, params={}, item_key="workflows"))

def get_workflow_meta(gh: GitHubClient, full_name: str, workflow_identifier: str) -> Optional[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_identifier}"
    return gh.request_json("GET", url)

def fetch_repo_file_text(gh: GitHubClient, full_name: str, repo_path: str, ref: Optional[str]) -> Tuple[str, Optional[int]]:
    url = f"https://api.github.com/repos/{full_name}/contents/{repo_path.lstrip('/')}"
    params = {"ref": ref} if ref else None
    data = gh.request_json("GET", url, params=params)
    if not data or not isinstance(data, dict):
        return "", None

    size = None
    try:
        if data.get("size") is not None:
            size = int(data.get("size"))
    except Exception:
        size = None

    if data.get("encoding") == "base64" and data.get("content"):
        return (b64_to_text(data["content"]) or ""), size

    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return (r.text or ""), size
        except requests.exceptions.RequestException:
            return "", size

    return "", size


# ============================================================
# V16-aligned preprocessing + patterns (compact but aligned to your miner)
# ============================================================
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$', '', text)
    text = re.sub(
        r'(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)',
        r'\1', text
    )
    return text

IGNORE_GHA_ACTIONS_RE = re.compile(
    r'(?mi)^\s*uses\s*:\s*('
    r'docker/(?:setup-qemu-action|setup-buildx-action|build-push-action|login-action)'
    r'|actions/checkout'
    r'|docker/setup-qemu-action'
    r'|docker/setup-buildx-action'
    r')@.*$'
)
def strip_irrelevant_ci_lines(text: str) -> str:
    return IGNORE_GHA_ACTIONS_RE.sub('', text or '')

EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def pre_sanitize(text: str) -> str:
    t = EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")
    return GHA_EXPR_RE.sub("", t or "")

# Env (styles)
REAL_DEVICE_ADB_RE = re.compile(r"(?mi)\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b")

EMU_COMMUNITY_ACTION_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*("
    r"reactivecircus/android-emulator-runner|"
    r"malinskiy/action-android/emulator-run-cmd|"
    r"hannesa2/action-android/emulator-run-cmd|"
    r"vgaidarji/android-github-actions-emulator"
    r")@"
)
OTHER_GHA_EMULATOR_ACTION_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*"
    r"(?!reactivecircus/android-emulator-runner@)"
    r"(?!malinskiy/action-android/emulator-run-cmd@)"
    r"(?!hannesa2/action-android/emulator-run-cmd@)"
    r"(?!emulator-wtf/run-tests@)"
    r"[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@"
)
EMU_CUSTOM_RUNTIME_RE = re.compile(
    r"(?mi)\b("
    r"emulator\b[^\n]*-avd\s+\S+|"
    r"adb\s+wait[- ]?for[- ]?device\b|"
    r"adb\s+-s\s+emulator-\d+\b|"
    r"\bandroid-wait-for-emulator\b|"
    r"\bstart-emulator\.sh\b|"
    r"\bavdmanager\b|"
    r"\bandroid\b[^\n]*\bcreate\s+avd\b"
    r")\b"
)

THIRD_PARTY_ENV_RE = re.compile(
    r"(?mi)\b("
    r"gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|"
    r"\bappcenter\s+test\s+run\s+android\b|"
    r"\bsaucectl\b|"
    r"\b(browserstack|bstack)\b|"
    r"\bmaestro\s+cloud\b|"
    r"\bemulator\.wtf\b|"
    r"^\s*uses\s*:\s*emulator-wtf/run-tests@"
    r")\b"
)

# Invocation
GRADLE_CONNECTED_RE = re.compile(r"(?mi)\b(connectedAndroidTest|connectedCheck|deviceCheck|allDevicesCheck|connectedBenchmarkAndroidTest)\b")
GRADLE_GMD_RE = re.compile(r"(?mi)\b(managedDeviceCheck|managedDevice)\b")  # allDevicesCheck intentionally NOT here
GRADLE_BASELINEPROFILE_RE = re.compile(r"(?mi)\b(generate\w*baselineprofile|collect\w*baselineprofile)\b")
ADB_AM_INSTR_RE = re.compile(r"(?mi)\bam\s+instrument\b")
THIRD_PARTY_INVOKE_RE = re.compile(
    r"(?mi)\b("
    r"gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|"
    r"\bflank\s+android\s+run\b|"
    r"\bappcenter\s+test\s+run\s+android\b|"
    r"\bsaucectl\b|"
    r"\b(browserstack|bstack)\b|"
    r"\bmaestro\s+cloud\b|"
    r"\bemulator\.wtf\b|"
    r"^\s*uses\s*:\s*emulator-wtf/run-tests@"
    r")\b"
)

# Called-file following (GitHub-side)
LOCAL_USES_RE = re.compile(r'(?mi)^\s*uses\s*:\s*(?P<ref>\./\S+?)(?:\s+#.*)?$')
WORKDIR_RE = re.compile(r'(?mi)^\s*working-directory\s*:\s*(?P<wd>[^\n#]+)')
SCRIPT_CALL_RE = re.compile(r'''(?mix)
(?:^|[;&|()\s"'`])
(?:(?:bash|sh|pwsh|powershell|python|python3|node|ruby)\s+)?(?P<path>(?:\./|\.\\)?[\w./\\-]+\.(?:sh|ps1|bat|cmd|py|js|rb|pl))
(?:\s|$)
''')
CONFIG_ARG_RE = re.compile(r'(?mi)\b--config(?:=|\s+)(?P<path>[^\s"\']+)')
NO_FOLLOW_BASENAMES = {"gradlew", "gradlew.bat", "gradle", "adb", "flutter"}

def _strip_quotes(s: str) -> str:
    return (s or "").strip().strip('"').strip("'").strip("`")

def is_dynamic_ref(ref: str) -> bool:
    r = ref or ""
    return ("${{" in r) or ("${" in r) or ("$(" in r) or ("%{" in r)

def extract_workdirs(text: str) -> List[str]:
    out = []
    for m in WORKDIR_RE.finditer(text or ""):
        wd = _strip_quotes(m.group("wd"))
        if wd:
            wd = wd.replace("\\", "/").lstrip("./").strip("/")
            out.append(wd)
    return unique_preserve(out)

def extract_references(text: str) -> List[str]:
    refs: List[str] = []
    for m in LOCAL_USES_RE.finditer(text or ""):
        ref = _strip_quotes(m.group("ref"))
        if "@" in ref:
            ref = ref.split("@", 1)[0]
        refs.append(ref)
    for m in SCRIPT_CALL_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))
    for m in CONFIG_ARG_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))

    out = []
    for r in refs:
        if not r:
            continue
        base = Path(r.replace("\\", "/")).name.lower()
        if base in NO_FOLLOW_BASENAMES:
            continue
        out.append(r.replace("\\", "/").strip())
    return unique_preserve(out)

def resolve_repo_paths(ref: str, workdirs: List[str]) -> List[str]:
    if not ref or is_dynamic_ref(ref):
        return []
    r = _strip_quotes(ref.replace("\\", "/").strip())
    if r.startswith("./"):
        r = r[2:]
    r = r.lstrip("/")

    prefixes = [""] + [wd.strip("/") for wd in (workdirs or []) if wd]
    candidates: List[str] = []
    for pref in prefixes:
        p = f"{pref}/{r}".strip("/") if pref else r
        candidates.append(p.strip("/"))

    out: List[str] = []
    for p in candidates:
        suffix = Path(p).suffix.lower()
        if not suffix:
            out.append(f"{p}/action.yml")
            out.append(f"{p}/action.yaml")
        out.append(p)
    return unique_preserve([x for x in out if x])

_GH_TEXT_CACHE: Dict[Tuple[str, str, str], str] = {}
def gh_fetch_text_cached(gh: GitHubClient, full_name: str, path: str, ref: Optional[str]) -> str:
    key = (full_name, path, ref or "")
    if key in _GH_TEXT_CACHE:
        return _GH_TEXT_CACHE[key]
    text, size = fetch_repo_file_text(gh, full_name, path, ref)
    if size is not None and size > MAX_FOLLOW_BYTES_GH:
        text = ""
    _GH_TEXT_CACHE[key] = text or ""
    return _GH_TEXT_CACHE[key]

def scan_text_v16_with_details(text: str) -> Tuple[Set[str], Set[str], Set[str]]:
    """
    Returns:
      styles: {Emu_Custom, Emu_Community, Third-Party, Real Device, (optional) GMD}
      inv: {Gradle_GMD, Gradle_Connected, Gradle, ADB, 3P CLIs}
      evidence: low-level matched buckets (for debugging/traceability)
    """
    raw = text or ""
    content = strip_comments(raw)
    content = normalize_block_keys(content)
    content = strip_irrelevant_ci_lines(content)
    content = pre_sanitize(content)
    low = content.lower()

    styles: Set[str] = set()
    inv: Set[str] = set()
    evidence: Set[str] = set()

    # styles
    if EMU_CUSTOM_RUNTIME_RE.search(low):
        styles.add("Emu_Custom")
        evidence.add("emu_custom_runtime")
    if EMU_COMMUNITY_ACTION_RE.search(low) or OTHER_GHA_EMULATOR_ACTION_RE.search(low):
        styles.add("Emu_Community")
        evidence.add("emu_community_action")
    if REAL_DEVICE_ADB_RE.search(low):
        styles.add("Real Device")
        evidence.add("real_device_adb")
    if THIRD_PARTY_ENV_RE.search(low):
        styles.add("Third-Party")
        evidence.add("third_party_env")

    # invocations
    if GRADLE_GMD_RE.search(low):
        inv.add("Gradle_GMD")
        evidence.add("gradle_gmd")
    if GRADLE_CONNECTED_RE.search(low):
        inv.add("Gradle_Connected")
        evidence.add("gradle_connected")
    if GRADLE_BASELINEPROFILE_RE.search(low):
        inv.add("Gradle")
        evidence.add("gradle_baselineprofile")
    if ADB_AM_INSTR_RE.search(low):
        inv.add("ADB")
        evidence.add("adb_am_instrument")
    if THIRD_PARTY_INVOKE_RE.search(low):
        inv.add("3P CLIs")
        evidence.add("third_party_invoke")

    # optional tag
    if "Gradle_GMD" in inv:
        styles.add("GMD")

    return styles, inv, evidence

# basic job splitter (good enough for workflow YAMLs)
JOBS_ANCHOR_RE = re.compile(r'(?m)^(?P<indent>\s*)jobs\s*:\s*$')
ANY_KEY_RE     = re.compile(r'(?m)^(?P<indent>\s*)(?P<name>[\w-]+)\s*:\s*$')

def split_jobs_blocks(raw: str) -> List[Tuple[str, str]]:
    m = JOBS_ANCHOR_RE.search(raw or "")
    if not m:
        return [("__whole__", raw or "")]
    jobs_indent = len(m.group("indent"))
    lines = (raw or "").splitlines(True)
    start_idx = (raw or "")[:m.end()].count("\n")
    candidates = []
    for i in range(start_idx, len(lines)):
        lm = ANY_KEY_RE.match(lines[i])
        if not lm:
            continue
        indent = len(lm.group("indent"))
        if indent > jobs_indent:
            candidates.append((i, indent, lm.group("name")))
    if not candidates:
        return [("__whole__", raw or "")]
    min_indent = min(indent for _, indent, _ in candidates)
    job_headers = [(i, name) for (i, indent, name) in candidates if indent == min_indent]
    if not job_headers:
        return [("__whole__", raw or "")]
    blocks = []
    header_indices = [i for i, _ in job_headers] + [len(lines)]
    for idx in range(len(job_headers)):
        i, name = job_headers[idx]
        j = header_indices[idx + 1]
        blocks.append((name, "".join(lines[i:j])))
    return blocks

def classify_workflow_v16_verified(
    gh: GitHubClient,
    full_name: str,
    wf_yaml_text: str,
    ref: Optional[str],
    follow_called_files: bool = True,
    max_depth: int = 2,
) -> Tuple[Set[str], Set[str], bool, Set[str], int, int]:
    """
    Returns:
      styles, inv, looks_like, evidence_labels, followed_files_count, unresolved_dynamic_refs_count
    """
    agg_styles: Set[str] = set()
    agg_inv: Set[str] = set()
    agg_evidence: Set[str] = set()

    # scan by job blocks
    for _, block in split_jobs_blocks(wf_yaml_text or ""):
        s, inv, ev = scan_text_v16_with_details(block)
        agg_styles |= s
        agg_inv |= inv
        agg_evidence |= ev

    followed_files_count = 0
    unresolved_dynamic_refs_count = 0

    if follow_called_files and wf_yaml_text:
        visited: Set[str] = set()
        queue: List[Tuple[str, int, List[str]]] = []

        refs_from_yaml = extract_references(wf_yaml_text)
        workdirs_from_yaml = extract_workdirs(wf_yaml_text)

        for r in refs_from_yaml:
            if is_dynamic_ref(r):
                unresolved_dynamic_refs_count += 1
                continue
            for p in resolve_repo_paths(r, workdirs_from_yaml):
                queue.append((p, 0, workdirs_from_yaml))

        while queue:
            path, depth, wds = queue.pop(0)
            if depth > max_depth or not path or path in visited:
                continue
            visited.add(path)

            txt = gh_fetch_text_cached(gh, full_name, path, ref=ref)
            if not txt:
                continue

            followed_files_count += 1

            s, inv, ev = scan_text_v16_with_details(txt)
            agg_styles |= s
            agg_inv |= inv
            agg_evidence |= ev

            if depth == max_depth:
                continue

            child_refs = extract_references(txt)
            child_wds = extract_workdirs(txt)

            for r in child_refs:
                if is_dynamic_ref(r):
                    unresolved_dynamic_refs_count += 1
                    continue
                for p in resolve_repo_paths(r, child_wds):
                    queue.append((p, depth + 1, child_wds))

    looks_like = bool(agg_styles or agg_inv)
    if not agg_inv:
        agg_inv.add("UNKNOWN")

    return agg_styles, agg_inv, looks_like, agg_evidence, followed_files_count, unresolved_dynamic_refs_count


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=3)
    gh = GitHubClient(tokens)

    repos = load_repo_fullnames_from_url_list_folder(URL_LIST_DIR, URL_LIST_GLOB)
    if not repos:
        raise RuntimeError(f"No repos found in URL list CSVs under: {URL_LIST_DIR}")

    out_fields = [
        "full_name",
        "default_branch",
        "has_any_gha_workflow",
        "workflow_identifier",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "workflow_html_url",
        "looks_like_instru",
        "styles",
        "invocation_types",
        "evidence_labels",
        "followed_files_count",
        "unresolved_dynamic_refs_count",
        "scanned_at_utc",
    ]
    ensure_csv_header(OUT_VERIFIED_WORKFLOWS_CSV, out_fields)

    # to avoid duplicates if rerun
    existing = load_existing_keys(OUT_VERIFIED_WORKFLOWS_CSV, "workflow_id")

    repo_iter = repos
    if tqdm is not None:
        repo_iter = tqdm(repos, desc="Repos (URL list -> verify workflows)")

    default_branch_cache: Dict[str, str] = {}

    for full_name in repo_iter:
        # default branch
        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        ref = default_branch if DEFAULT_BRANCH_ONLY else None

        workflows = list_workflows(gh, full_name) or []
        has_any = "yes" if workflows else "no"
        if not workflows:
            # only keep repos that have at least one workflow file (your requirement)
            continue

        for wf in workflows:
            wf_id = str(wf.get("id") or "").strip()
            if wf_id and wf_id in existing:
                continue

            wf_name = (wf.get("name") or "").strip()
            wf_path = (wf.get("path") or "").strip()
            wf_html = (wf.get("html_url") or "").strip()
            wf_identifier = (wf_path.split("/")[-1] if wf_path else str(wf.get("path") or ""))

            # get meta (to confirm path/name; optional)
            meta = get_workflow_meta(gh, full_name, wf_identifier) or {}
            if isinstance(meta, dict):
                wf_name = (meta.get("name") or wf_name).strip()
                wf_path = (meta.get("path") or wf_path).strip()
                wf_html = (meta.get("html_url") or wf_html).strip()
                wf_id = str(meta.get("id") or wf_id).strip()

            if not wf_path:
                continue

            yml_text, size = fetch_repo_file_text(gh, full_name, wf_path, ref=ref)
            if size is not None and size > MAX_FOLLOW_BYTES_GH:
                yml_text = ""

            styles, inv, looks_like, ev_labels, followed_cnt, unresolved_cnt = classify_workflow_v16_verified(
                gh=gh,
                full_name=full_name,
                wf_yaml_text=yml_text or "",
                ref=ref,
                follow_called_files=FOLLOW_CALLED_FILES_GH,
                max_depth=MAX_FOLLOW_DEPTH_GH,
            )

            append_row(OUT_VERIFIED_WORKFLOWS_CSV, out_fields, {
                "full_name": full_name,
                "default_branch": default_branch,
                "has_any_gha_workflow": has_any,
                "workflow_identifier": wf_identifier,
                "workflow_id": wf_id,
                "workflow_name": wf_name,
                "workflow_path": wf_path,
                "workflow_html_url": wf_html,
                "looks_like_instru": "yes" if looks_like else "no",
                "styles": ",".join(sorted(styles)),
                "invocation_types": ",".join(sorted(inv)),
                "evidence_labels": ",".join(sorted(ev_labels)),
                "followed_files_count": str(followed_cnt),
                "unresolved_dynamic_refs_count": str(unresolved_cnt),
                "scanned_at_utc": now_utc_iso(),
            })

            if wf_id:
                existing.add(wf_id)

    print("Done.")
    print("Verified workflows:", OUT_VERIFIED_WORKFLOWS_CSV)


if __name__ == "__main__":
    main()


Repos (URL list -> verify workflows): 100%|██████████| 481/481 [13:13<00:00,  1.65s/it]

Done.
Verified workflows: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\verified_workflows_v16.csv


In [ ]:
#Stage 2: run metrics extraction + attach env_style/style labels to each run row

In [3]:
### repeated Stage 2 with complete feed to Stage 3: includes some of the useful instru style related columns from stage 1

In [4]:
# ============================================================
# Stage 2: Run-metrics extraction (from verified_workflows_v16.csv)
#
# Input :
#   verified_workflows_v16.csv  (output of Stage 1)
# Output:
#   run_metrics_v16_from_verified_workflows.csv
#
# Change in this version:
#   ✅ ONLY adds workflow-level columns from verified_workflows_v16.csv into every run row:
#      - looks_like_instru
#      - styles
#      - invocation_types
#      - evidence_labels   (if present in verified CSV; blank otherwise)
#      - label_source      (if present; blank otherwise)
#
# Nothing else changed.
# ============================================================

import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# ============================================================
# CONFIG
# ============================================================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext"
)

IN_VERIFIED_WORKFLOWS_CSV = ROOT_DIR / "verified_workflows_v16.csv"
OUT_RUN_METRICS_CSV = ROOT_DIR / "run_metrics_v16_from_verified_workflows_with_env_style.csv"

DEFAULT_BRANCH_ONLY = True
PROCESS_ONLY_LOOKS_LIKE_INSTRU = True  # set False if you want runs for all workflows
FETCH_JOBS_FOR_EACH_RUN = True

MAX_RUNS_PER_WORKFLOW: Optional[int] = None
RUN_CREATED_AT_AFTER: Optional[str] = None  # ISO8601 or None

# ----------- API robustness knobs -----------
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# Use ONLY token 1..3
MAX_TOKENS_TO_USE = 3


# ============================================================
# Helpers
# ============================================================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    if csv_path.exists():
        return
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join_names(names: List[str], max_len: int = 500) -> str:
    s = ",".join(unique_preserve([n.strip() for n in names if n and n.strip()]))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."


# ============================================================
# GitHub API client (token rotation + retries)
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-metrics-v16/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# ============================================================
# GitHub endpoints
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_workflow_runs(gh: GitHubClient, full_name: str, workflow_identifier: str, branch: Optional[str]) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_identifier}/runs"
    params = {"branch": branch} if branch else {}
    return list(gh.paginate(url, params=params, item_key="workflow_runs"))

def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


# ============================================================
# Instrumentation detection inside runs (jobs/steps)
# ============================================================
INSTRU_STEP_NAME_RE = re.compile(
    r"(instrument|connected.*androidtest|androidtest|manageddevice|gmd|emulator runner|"
    r"firebase test|test lab|device farm|uiautomator|espresso)",
    re.IGNORECASE,
)
INSTRU_JOB_NAME_RE = re.compile(
    r"(instrument|androidtest|connected|manageddevice|gmd|emulator|firebase|test lab|device farm)",
    re.IGNORECASE,
)

def infer_instru_metrics_from_jobs(
    jobs: List[Dict],
    run_created_at: str,
    run_started_at: str,
) -> Dict[str, Union[str, int, float, None]]:
    """
    Returns a dict matching Stage-2 output columns.
    """
    out = {
        "instru_conclusion": "unknown",
        "instru_detect_method": "none",
        "instru_duration_seconds": None,
        "run_duration_seconds": None,
        "runner_labels_union": "",
        "queue_seconds": None,
        "time_to_first_instru_seconds": None,
        "instru_job_count": 0,
        "instru_step_count": 0,
        "instru_job_names": "",
        "instru_step_names": "",
        "instru_total_seconds": None,
        "instru_window_seconds": None,
        "instru_first_started_at": "",
        "instru_last_completed_at": "",
        "instru_share_of_run": None,
    }

    out["queue_seconds"] = dt_to_seconds(iso_to_dt(run_created_at), iso_to_dt(run_started_at))

    if not jobs:
        return out

    starts, ends = [], []
    labels_union: Set[str] = set()
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt: starts.append(sdt)
        if edt: ends.append(edt)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                labels_union.add(lab.strip())
    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)
    out["run_duration_seconds"] = run_dur
    out["runner_labels_union"] = ",".join(sorted(labels_union))

    instru_job_names: List[str] = []
    instru_step_names: List[str] = []

    instru_first_start: Optional[datetime] = None
    instru_last_end: Optional[datetime] = None

    total_seconds = 0
    total_seconds_any = False

    first_match_set = False

    for j in jobs:
        job_name = (j.get("name") or "").strip()
        job_is_instru = bool(INSTRU_JOB_NAME_RE.search(job_name))
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        step_matches = []
        for st in steps:
            step_name = (st.get("name") or "").strip()
            if step_name and INSTRU_STEP_NAME_RE.search(step_name):
                step_matches.append(st)

        if job_is_instru or step_matches:
            if job_name:
                instru_job_names.append(job_name)

        for st in step_matches:
            step_name = (st.get("name") or "").strip()
            if step_name:
                instru_step_names.append(step_name)

            sdt = iso_to_dt(st.get("started_at")) or iso_to_dt(j.get("started_at"))
            edt = iso_to_dt(st.get("completed_at")) or iso_to_dt(j.get("completed_at"))

            if sdt and (instru_first_start is None or sdt < instru_first_start):
                instru_first_start = sdt
            if edt and (instru_last_end is None or edt > instru_last_end):
                instru_last_end = edt

            dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
            if dur is None:
                dur = dt_to_seconds(iso_to_dt(j.get("started_at")), iso_to_dt(j.get("completed_at")))
            if dur is not None:
                total_seconds += dur
                total_seconds_any = True

            if not first_match_set:
                out["instru_conclusion"] = (st.get("conclusion") or st.get("status") or "unknown")
                out["instru_detect_method"] = "step"
                out["instru_duration_seconds"] = dur
                first_match_set = True

        if (job_is_instru and not step_matches):
            sdt = iso_to_dt(j.get("started_at"))
            edt = iso_to_dt(j.get("completed_at"))

            if sdt and (instru_first_start is None or sdt < instru_first_start):
                instru_first_start = sdt
            if edt and (instru_last_end is None or edt > instru_last_end):
                instru_last_end = edt

            dur = dt_to_seconds(sdt, edt)
            if dur is not None:
                total_seconds += dur
                total_seconds_any = True

            if not first_match_set:
                out["instru_conclusion"] = (j.get("conclusion") or "unknown")
                out["instru_detect_method"] = "job"
                out["instru_duration_seconds"] = dur
                first_match_set = True

    out["instru_job_names"] = safe_join_names(instru_job_names)
    out["instru_step_names"] = safe_join_names(instru_step_names)
    out["instru_job_count"] = len(unique_preserve(instru_job_names))
    out["instru_step_count"] = len(unique_preserve(instru_step_names))

    if total_seconds_any:
        out["instru_total_seconds"] = total_seconds

    if instru_first_start:
        out["instru_first_started_at"] = instru_first_start.isoformat()
        base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)
        out["time_to_first_instru_seconds"] = dt_to_seconds(base_start, instru_first_start)

    if instru_last_end:
        out["instru_last_completed_at"] = instru_last_end.isoformat()

    out["instru_window_seconds"] = dt_to_seconds(instru_first_start, instru_last_end)

    if out["instru_window_seconds"] is not None and run_dur:
        try:
            out["instru_share_of_run"] = round(float(out["instru_window_seconds"]) / float(run_dur), 6)
        except Exception:
            out["instru_share_of_run"] = None

    return out


# ============================================================
# Read verified workflows
# ============================================================
def load_verified_workflows(path: Path) -> List[Dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(f"Verified workflows CSV not found: {path}")
    rows: List[Dict[str, str]] = []
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            rows.append({k: (v or "") for k, v in r.items()})
    return rows


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    # --- overwrite outputs to guarantee one-shot reproducibility ---
    if OUT_RUN_METRICS_CSV.exists():
        OUT_RUN_METRICS_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    rows = load_verified_workflows(IN_VERIFIED_WORKFLOWS_CSV)
    if PROCESS_ONLY_LOOKS_LIKE_INSTRU:
        rows = [r for r in rows if (r.get("looks_like_instru", "").strip().lower() == "yes")]

    if not rows:
        raise RuntimeError("No workflows found to process (check verified CSV or filter).")

    after_dt = iso_to_dt(RUN_CREATED_AT_AFTER) if RUN_CREATED_AT_AFTER else None

    # ✅ Added workflow-level columns here (no other changes)
    out_fields = [
        "full_name",
        "default_branch",
        "workflow_identifier",
        "workflow_id",
        "workflow_name",
        "workflow_path",

        # NEW: carry over workflow-level labels/evidence into run rows
        "looks_like_instru",
        "styles",
        "invocation_types",
        "evidence_labels",
        "label_source",

        "run_id",
        "run_number",
        "run_attempt",
        "head_sha",
        "created_at",
        "run_started_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "html_url",
        "extracted_at_utc",
        # performance / instru
        "queue_seconds",
        "time_to_first_instru_seconds",
        "instru_conclusion",
        "instru_detect_method",
        "instru_duration_seconds",
        "run_duration_seconds",
        "runner_labels_union",
        "instru_job_count",
        "instru_step_count",
        "instru_job_names",
        "instru_step_names",
        "instru_total_seconds",
        "instru_window_seconds",
        "instru_first_started_at",
        "instru_last_completed_at",
        "instru_share_of_run",
    ]

    ensure_csv_header(OUT_RUN_METRICS_CSV, out_fields)
    existing_run_ids = load_existing_keys(OUT_RUN_METRICS_CSV, "run_id")

    default_branch_cache: Dict[str, str] = {}

    wf_iter = rows
    if tqdm is not None:
        wf_iter = tqdm(rows, desc="Workflows -> runs -> jobs")

    for wf in wf_iter:
        full_name = (wf.get("full_name") or "").strip()
        workflow_identifier = (wf.get("workflow_identifier") or "").strip()
        workflow_id = (wf.get("workflow_id") or "").strip()
        workflow_name = (wf.get("workflow_name") or "").strip()
        workflow_path = (wf.get("workflow_path") or "").strip()

        if not full_name or not workflow_identifier:
            continue

        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        branch = default_branch if DEFAULT_BRANCH_ONLY else None

        runs = list_workflow_runs(gh, full_name, workflow_identifier, branch=branch) or []
        if MAX_RUNS_PER_WORKFLOW is not None:
            runs = runs[:MAX_RUNS_PER_WORKFLOW]

        for run in runs:
            run_id = str(run.get("id") or "").strip()
            if not run_id or run_id in existing_run_ids:
                continue

            created_at = run.get("created_at") or ""
            if after_dt:
                cdt = iso_to_dt(created_at)
                if cdt and cdt < after_dt:
                    continue

            head_branch = run.get("head_branch") or ""
            if DEFAULT_BRANCH_ONLY and head_branch and head_branch != default_branch:
                continue

            run_started_at = run.get("run_started_at") or ""
            head_sha = run.get("head_sha") or ""

            metrics = infer_instru_metrics_from_jobs(
                jobs=list_run_jobs(gh, full_name, int(run_id)) if FETCH_JOBS_FOR_EACH_RUN else [],
                run_created_at=created_at,
                run_started_at=run_started_at,
            )

            append_row(OUT_RUN_METRICS_CSV, out_fields, {
                "full_name": full_name,
                "default_branch": default_branch,
                "workflow_identifier": workflow_identifier,
                "workflow_id": workflow_id,
                "workflow_name": workflow_name,
                "workflow_path": workflow_path,

                # ✅ NEW: write-through from verified workflow row
                "looks_like_instru": (wf.get("looks_like_instru") or ""),
                "styles": (wf.get("styles") or ""),
                "invocation_types": (wf.get("invocation_types") or ""),
                "evidence_labels": (wf.get("evidence_labels") or ""),
                "label_source": (wf.get("label_source") or ""),

                "run_id": run_id,
                "run_number": run.get("run_number") or "",
                "run_attempt": run.get("run_attempt") or "",
                "head_sha": head_sha,
                "created_at": created_at,
                "run_started_at": run_started_at,
                "status": run.get("status") or "",
                "run_conclusion": run.get("conclusion") or "",
                "event": run.get("event") or "",
                "head_branch": head_branch,
                "html_url": run.get("html_url") or "",
                "extracted_at_utc": now_utc_iso(),
                "queue_seconds": "" if metrics["queue_seconds"] is None else str(metrics["queue_seconds"]),
                "time_to_first_instru_seconds": "" if metrics["time_to_first_instru_seconds"] is None else str(metrics["time_to_first_instru_seconds"]),
                "instru_conclusion": metrics["instru_conclusion"],
                "instru_detect_method": metrics["instru_detect_method"],
                "instru_duration_seconds": "" if metrics["instru_duration_seconds"] is None else str(metrics["instru_duration_seconds"]),
                "run_duration_seconds": "" if metrics["run_duration_seconds"] is None else str(metrics["run_duration_seconds"]),
                "runner_labels_union": metrics["runner_labels_union"],
                "instru_job_count": str(metrics["instru_job_count"]),
                "instru_step_count": str(metrics["instru_step_count"]),
                "instru_job_names": metrics["instru_job_names"],
                "instru_step_names": metrics["instru_step_names"],
                "instru_total_seconds": "" if metrics["instru_total_seconds"] is None else str(metrics["instru_total_seconds"]),
                "instru_window_seconds": "" if metrics["instru_window_seconds"] is None else str(metrics["instru_window_seconds"]),
                "instru_first_started_at": metrics["instru_first_started_at"],
                "instru_last_completed_at": metrics["instru_last_completed_at"],
                "instru_share_of_run": "" if metrics["instru_share_of_run"] is None else str(metrics["instru_share_of_run"]),
            })

            existing_run_ids.add(run_id)

    print("Done.")
    print("Run metrics:", OUT_RUN_METRICS_CSV)


if __name__ == "__main__":
    main()


Workflows -> runs -> jobs: 100%|██████████| 357/357 [5:29:07<00:00, 55.32s/it]     

Done.
Run metrics: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\run_metrics_v16_from_verified_workflows_with_env_style.csv


In [5]:
#Stage 3 to expand the coverage of the key metrics

In [6]:
# ============================================================
# Stage 3 (FIXED + UPGRADED):
#  - Fixes BOM in CSV headers so rows are processed
#  - YAML-assisted step attribution (better Third-Party coverage)
#  - Outputs per-step breakdown for style-by-style comparisons
#
# Input :
#   run_metrics_v16_from_verified_workflows_with_env_style.csv  (Stage 2)
# Output:
#   run_metrics_v16_stage3_enhanced.csv
#   run_steps_v16_stage3_breakdown.csv
#
# Uses only GITHUB_TOKEN_1..3 by default
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext"
)

IN_STAGE2_CSV = ROOT_DIR / "run_metrics_v16_from_verified_workflows_with_env_style.csv"
OUT_STAGE3_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
OUT_STEPS_CSV  = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"

MAX_TOKENS_TO_USE = 3
PROCESS_ONLY_RELEVANT_ROWS = True  # set False to recompute for all rows

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# Fetch workflow YAML (recommended). If False, Stage3 behaves closer to your original.
FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000  # limit cache size

# =========================
# Helpers
# =========================
BOM = "\ufeff"

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join_names(names: List[str], max_len: int = 800) -> str:
    s = ",".join(unique_preserve([n.strip() for n in names if n and n.strip()]))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def _clean_key(k: str) -> str:
    # Strip BOM and whitespace from headers
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        # normalize headers
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                ck = _clean_key(k)
                clean_row[ck] = (v or "")
            rows.append(clean_row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def ensure_csv(path: Path, fieldnames: List[str]) -> None:
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(path: Path, fieldnames: List[str], row: Dict[str, str]) -> None:
    with path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-metrics-stage3-fixed/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))

def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    # contents API returns base64 or download_url
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

# =========================
# Patterns (names + YAML content)
# =========================
TEST_ANY_RE = re.compile(
    r"(am\s+instrument|connected.*androidtest|androidtest|manageddevice|gmd|devicecheck|"
    r"connectedcheck|espresso|uiautomator|firebase\s+test|test\s+lab|device\s+farm|flank|"
    r"appcenter\s+test|saucectl|browserstack|bstack|maestro\s+cloud|emulator\.wtf)",
    re.IGNORECASE,
)

ENV_ANY_RE = re.compile(
    r"(reactivecircus|android-emulator-runner|emulator\b|avd\b|avdmanager|sdkmanager|"
    r"start[-_ ]emulator|android-wait-for-emulator|adb\s+wait[- ]?for[- ]?device|kvm)",
    re.IGNORECASE,
)

THIRD_PARTY_ANY_RE = re.compile(
    r"(gcloud.*firebase\s+test\s+android\s+run|firebase\s+test\s+lab|flank\s+android\s+run|"
    r"appcenter\s+test\s+run\s+android|saucectl|browserstack|bstack|maestro\s+cloud|"
    r"emulator\.wtf)",
    re.IGNORECASE,
)

ARTIFACT_RE = re.compile(r"(upload[- ]artifact|actions/upload-artifact)", re.IGNORECASE)

def parse_yaml_step_hints(yaml_text: str) -> Dict[str, Dict[str, bool]]:
    """
    Very lightweight heuristic parser:
      builds { step_name_lower: {is_env,is_test,is_3p,is_artifact} }
    Works even without PyYAML.
    """
    hints: Dict[str, Dict[str, bool]] = {}
    if not yaml_text:
        return hints

    # capture "- name: X" blocks and scan following few lines for run/uses content
    lines = yaml_text.splitlines()
    i = 0
    while i < len(lines):
        line = lines[i]
        m = re.match(r"^\s*-\s*name\s*:\s*(.+?)\s*$", line)
        if not m:
            i += 1
            continue
        step_name = m.group(1).strip().strip('"').strip("'")
        key = step_name.lower()
        blob = "\n".join(lines[i : min(len(lines), i + 12)])  # lookahead window
        hints[key] = {
            "is_env": bool(ENV_ANY_RE.search(blob)),
            "is_test": bool(TEST_ANY_RE.search(blob)),
            "is_3p": bool(THIRD_PARTY_ANY_RE.search(blob)),
            "is_artifact": bool(ARTIFACT_RE.search(blob)),
        }
        i += 1
    return hints

# =========================
# Metrics + step breakdown
# =========================
def enhanced_metrics_and_steps(
    jobs: List[Dict],
    run_created_at: str,
    run_started_at: str,
    yaml_hints: Dict[str, Dict[str, bool]],
    styles_text: str,
) -> Tuple[Dict[str, Union[str, int, float, None]], List[Dict[str, str]]]:
    """
    Produces:
      - Stage3 metrics (core/extended + 3P job attribution)
      - Per-step breakdown rows (category + duration)
    """
    out = {
        "queue_seconds": dt_to_seconds(iso_to_dt(run_created_at), iso_to_dt(run_started_at)),
        "time_to_first_instru_seconds": None,

        # keep Stage2 semantics
        "instru_conclusion": "unknown",
        "instru_detect_method": "none",
        "instru_duration_seconds": None,

        "run_duration_seconds": None,
        "runner_labels_union": "",

        "instru_job_count": 0,
        "instru_step_count": 0,
        "instru_job_names": "",
        "instru_step_names": "",
        "instru_total_seconds": None,
        "instru_window_seconds": None,
        "instru_first_started_at": "",
        "instru_last_completed_at": "",
        "instru_share_of_run": None,

        # new
        "instru_extended_window_seconds": None,
        "env_setup_seconds": None,               # (first test start - first env start)
        "env_setup_sum_seconds": None,           # sum of env steps durations
        "artifact_sum_seconds": None,            # sum of artifact steps durations
        "third_party_job_count": 0,
        "third_party_job_names": "",
    }

    step_rows: List[Dict[str, str]] = []

    if not jobs:
        return out, step_rows

    # Run duration + labels
    starts, ends = [], []
    labels_union: Set[str] = set()
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt: starts.append(sdt)
        if edt: ends.append(edt)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                labels_union.add(lab.strip())
    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)
    out["run_duration_seconds"] = run_dur
    out["runner_labels_union"] = ",".join(sorted(labels_union))

    # Boundaries
    core_first: Optional[datetime] = None
    core_last: Optional[datetime] = None
    ext_first: Optional[datetime] = None
    ext_last: Optional[datetime] = None

    first_env_start: Optional[datetime] = None
    first_test_start: Optional[datetime] = None

    total_test = 0
    total_env = 0
    total_art = 0
    total_any = False

    instru_job_names: List[str] = []
    instru_step_names: List[str] = []
    third_party_job_names: List[str] = []
    third_party_count = 0

    first_match_set = False

    # Style hint: if workflow style says Third-Party, be more permissive
    style_third_party = "third-party" in (styles_text or "").lower()

    for j in jobs:
        job_id = str(j.get("id") or "")
        job_name = (j.get("name") or "").strip()
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        job_start = iso_to_dt(j.get("started_at"))
        job_end = iso_to_dt(j.get("completed_at"))
        job_dur = dt_to_seconds(job_start, job_end)

        # 3P detection: job name, step name, OR style hint + any test hint inside steps
        job_is_3p = bool(THIRD_PARTY_ANY_RE.search(job_name)) or any(
            THIRD_PARTY_ANY_RE.search((st.get("name") or "")) for st in steps
        )

        # If YAML hints identify this job as third-party by step name, include it
        if not job_is_3p:
            for st in steps:
                sn = (st.get("name") or "").strip().lower()
                if sn and sn in yaml_hints and yaml_hints[sn].get("is_3p"):
                    job_is_3p = True
                    break

        if style_third_party and not job_is_3p:
            # If the workflow is Third-Party style, treat any step with TEST hints as 3P-related
            if any(TEST_ANY_RE.search((st.get("name") or "")) for st in steps):
                job_is_3p = True

        if job_is_3p:
            third_party_count += 1
            if job_name:
                third_party_job_names.append(job_name)
                instru_job_names.append(job_name)

            # Attribute full job duration as "test" (core+extended), because the real work is remote
            if job_start and (core_first is None or job_start < core_first): core_first = job_start
            if job_end   and (core_last  is None or job_end   > core_last):  core_last  = job_end
            if job_start and (ext_first  is None or job_start < ext_first):  ext_first  = job_start
            if job_end   and (ext_last   is None or job_end   > ext_last):   ext_last   = job_end

            if job_dur is not None:
                total_test += job_dur
                total_any = True
                if not first_match_set:
                    out["instru_conclusion"] = (j.get("conclusion") or "unknown")
                    out["instru_detect_method"] = "third_party_job"
                    out["instru_duration_seconds"] = job_dur
                    first_match_set = True

            # Still emit per-step rows (categorized as third_party)
            for st in steps:
                sname = (st.get("name") or "").strip()
                sdt = iso_to_dt(st.get("started_at")) or job_start
                edt = iso_to_dt(st.get("completed_at")) or job_end
                dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
                step_rows.append({
                    "job_id": job_id,
                    "job_name": job_name,
                    "step_name": sname,
                    "category": "third_party",
                    "started_at": sdt.isoformat() if sdt else "",
                    "completed_at": edt.isoformat() if edt else "",
                    "duration_seconds": "" if dur is None else str(dur),
                })

            continue

        # Non-3P path: step classification (names + YAML hints)
        job_is_instruish = bool(TEST_ANY_RE.search(job_name) or ENV_ANY_RE.search(job_name))
        if not job_is_instruish:
            job_is_instruish = any(
                TEST_ANY_RE.search((st.get("name") or "")) or ENV_ANY_RE.search((st.get("name") or ""))
                for st in steps
            )
        if not job_is_instruish:
            for st in steps:
                sn = (st.get("name") or "").strip().lower()
                if sn and sn in yaml_hints and (yaml_hints[sn].get("is_env") or yaml_hints[sn].get("is_test")):
                    job_is_instruish = True
                    break
        if not job_is_instruish:
            continue

        if job_name:
            instru_job_names.append(job_name)

        for st in steps:
            step_name = (st.get("name") or "").strip()
            if not step_name:
                continue

            sn = step_name.lower()
            hint = yaml_hints.get(sn, {})

            is_env = bool(ENV_ANY_RE.search(step_name)) or bool(hint.get("is_env"))
            is_test = bool(TEST_ANY_RE.search(step_name)) or bool(hint.get("is_test"))
            is_art = bool(ARTIFACT_RE.search(step_name)) or bool(hint.get("is_artifact"))

            st_start = iso_to_dt(st.get("started_at")) or job_start
            st_end = iso_to_dt(st.get("completed_at")) or job_end
            st_dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
            if st_dur is None:
                st_dur = dt_to_seconds(job_start, job_end)

            # Category
            if is_art:
                cat = "artifact"
            elif is_test:
                cat = "test"
            elif is_env:
                cat = "env_setup"
            else:
                cat = "other"

            # Emit step row
            step_rows.append({
                "job_id": job_id,
                "job_name": job_name,
                "step_name": step_name,
                "category": cat,
                "started_at": st_start.isoformat() if st_start else "",
                "completed_at": st_end.isoformat() if st_end else "",
                "duration_seconds": "" if st_dur is None else str(st_dur),
            })

            # Extended window includes env+test
            if (is_env or is_test) and st_start and (ext_first is None or st_start < ext_first):
                ext_first = st_start
            if (is_env or is_test) and st_end and (ext_last is None or st_end > ext_last):
                ext_last = st_end

            # Track env/test first starts
            if is_env and st_start and (first_env_start is None or st_start < first_env_start):
                first_env_start = st_start
            if is_test and st_start and (first_test_start is None or st_start < first_test_start):
                first_test_start = st_start

            # Core window (test only)
            if is_test:
                instru_step_names.append(step_name)
                if st_start and (core_first is None or st_start < core_first): core_first = st_start
                if st_end   and (core_last  is None or st_end   > core_last):  core_last  = st_end
                if st_dur is not None:
                    total_test += st_dur
                    total_any = True
                if not first_match_set:
                    out["instru_conclusion"] = (st.get("conclusion") or st.get("status") or "unknown")
                    out["instru_detect_method"] = "step_yaml_assisted"
                    out["instru_duration_seconds"] = st_dur
                    first_match_set = True

            if is_env and st_dur is not None:
                total_env += st_dur
            if is_art and st_dur is not None:
                total_art += st_dur

    out["instru_job_names"] = safe_join_names(instru_job_names)
    out["instru_step_names"] = safe_join_names(instru_step_names)
    out["instru_job_count"] = len(unique_preserve(instru_job_names))
    out["instru_step_count"] = len(unique_preserve(instru_step_names))

    out["third_party_job_count"] = third_party_count
    out["third_party_job_names"] = safe_join_names(third_party_job_names)

    # Core window
    out["instru_first_started_at"] = core_first.isoformat() if core_first else ""
    out["instru_last_completed_at"] = core_last.isoformat() if core_last else ""
    out["instru_window_seconds"] = dt_to_seconds(core_first, core_last)

    # Time-to-first
    if core_first:
        base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)
        out["time_to_first_instru_seconds"] = dt_to_seconds(base_start, core_first)

    # Extended window
    out["instru_extended_window_seconds"] = dt_to_seconds(ext_first, ext_last)

    # env setup delta
    if first_env_start and first_test_start:
        out["env_setup_seconds"] = dt_to_seconds(first_env_start, first_test_start)

    # Totals
    if total_any:
        out["instru_total_seconds"] = total_test
    out["env_setup_sum_seconds"] = total_env if total_env > 0 else None
    out["artifact_sum_seconds"] = total_art if total_art > 0 else None

    # Share of run (core)
    if out["instru_window_seconds"] is not None and run_dur:
        try:
            out["instru_share_of_run"] = round(float(out["instru_window_seconds"]) / float(run_dur), 6)
        except Exception:
            out["instru_share_of_run"] = None

    return out, step_rows

# =========================
# MAIN
# =========================
def main() -> None:
    # --- overwrite outputs to guarantee one-shot reproducibility ---
    if OUT_STAGE3_CSV.exists():
        OUT_STAGE3_CSV.unlink()
    if OUT_STEPS_CSV.exists():
        OUT_STEPS_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    rows, in_fields = read_csv_rows(IN_STAGE2_CSV)
    if not rows:
        raise RuntimeError("No rows in Stage-2 input CSV.")

    # Relevance filter
    if PROCESS_ONLY_RELEVANT_ROWS:
        def is_relevant(r: Dict[str, str]) -> bool:
            det = (r.get("instru_detect_method", "") or "").strip().lower()
            styles = (r.get("styles", "") or "").lower()
            inv = (r.get("invocation_types", "") or "").lower()
            looks = (r.get("looks_like_instru", "") or "").strip().lower()
            return (
                looks == "yes"
                or det not in ("", "none", "unknown")
                or ("third-party" in styles)
                or ("3p clis" in inv)
            )
        target = [r for r in rows if is_relevant(r)]
    else:
        target = rows

    print(f"Rows total: {len(rows)} | Rows to enhance: {len(target)}")

    # Output fields: keep Stage2 + append new Stage3 fields
    stage3_cols = [
        "instru_detect_method",
        "instru_duration_seconds",
        "instru_total_seconds",
        "instru_window_seconds",
        "instru_extended_window_seconds",
        "env_setup_seconds",
        "env_setup_sum_seconds",
        "artifact_sum_seconds",
        "third_party_job_count",
        "third_party_job_names",
        "stage3_extracted_at_utc",
    ]

    out_fieldnames = list(in_fields)
    for c in stage3_cols:
        if c not in out_fieldnames:
            out_fieldnames.append(c)

    # Steps breakdown file schema
    steps_fields = [
        "full_name",
        "run_id",
        "workflow_identifier",
        "workflow_path",
        "head_sha",
        "styles",
        "job_id",
        "job_name",
        "step_name",
        "category",
        "started_at",
        "completed_at",
        "duration_seconds",
        "stage3_extracted_at_utc",
    ]
    ensure_csv(OUT_STEPS_CSV, steps_fields)

    # Workflow YAML cache (repo, path, sha) -> hints
    yaml_cache: Dict[Tuple[str, str, str], Dict[str, Dict[str, bool]]] = {}

    it = target
    if tqdm is not None:
        it = tqdm(target, desc="Stage3: enhance runs")

    for r in it:
        full_name = (r.get("full_name") or "").strip()
        run_id = (r.get("run_id") or "").strip()
        created_at = r.get("created_at") or ""
        run_started_at = r.get("run_started_at") or ""
        workflow_identifier = (r.get("workflow_identifier") or "").strip()
        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()
        styles_text = (r.get("styles") or "")

        if not full_name or not run_id:
            continue

        # YAML hints (optional)
        hints: Dict[str, Dict[str, bool]] = {}
        if FETCH_WORKFLOW_YAML and workflow_path and head_sha:
            ck = (full_name, workflow_path, head_sha)
            if ck in yaml_cache:
                hints = yaml_cache[ck]
            else:
                yml = fetch_workflow_yaml(gh, full_name, workflow_path, ref=head_sha)
                hints = parse_yaml_step_hints(yml)
                # keep cache bounded
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = hints

        jobs = list_run_jobs(gh, full_name, int(run_id)) or []

        m, step_rows = enhanced_metrics_and_steps(
            jobs=jobs,
            run_created_at=created_at,
            run_started_at=run_started_at,
            yaml_hints=hints,
            styles_text=styles_text,
        )

        # write back stage3 cols
        for k in stage3_cols:
            if k == "stage3_extracted_at_utc":
                r[k] = now_utc_iso()
            else:
                v = m.get(k)
                r[k] = "" if v is None else str(v)

        # append step breakdown rows
        ts = r.get("stage3_extracted_at_utc", now_utc_iso())
        for sr in step_rows:
            append_row(OUT_STEPS_CSV, steps_fields, {
                "full_name": full_name,
                "run_id": run_id,
                "workflow_identifier": workflow_identifier,
                "workflow_path": workflow_path,
                "head_sha": head_sha,
                "styles": styles_text,
                **sr,
                "stage3_extracted_at_utc": ts,
            })

    write_csv(OUT_STAGE3_CSV, out_fieldnames, rows)
    print("Done.")
    print("Stage 3 output:", OUT_STAGE3_CSV)
    print("Step breakdown:", OUT_STEPS_CSV)

if __name__ == "__main__":
    main()


Rows total: 31060 | Rows to enhance: 31060


Stage3: enhance runs:  92%|█████████▏| 28657/31060 [5:07:05<51:30:42, 77.17s/it]

[giveup] GET https://api.github.com/repos/pytorch/executorch/actions/runs/20148384379/jobs after 8 tries (last_status=502)


Stage3: enhance runs:  93%|█████████▎| 28768/31060 [5:21:35<48:57:27, 76.90s/it]

[giveup] GET https://api.github.com/repos/pytorch/executorch/actions/runs/19863006272/jobs after 8 tries (last_status=502)


Stage3: enhance runs: 100%|██████████| 31060/31060 [6:23:02<00:00,  1.35it/s]   


Done.
Stage 3 output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\run_metrics_v16_stage3_enhanced.csv
Step breakdown: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv


In [ ]:
## Improved Stage 3 to includes a few more helpful fields

In [1]:
# ============================================================
# Stage 3 (ADJUSTED for Item 1: TTFTS):
#  - Adds TTFTS (time-to-first-test-step start) as primary metric
#  - Keeps old time_to_first_instru_seconds as fallback/legacy
#  - Adds optional fields:
#       - first_test_step_started_at
#       - ttfts_seconds
#       - ttfts_source
#
# Input :
#   run_metrics_v16_from_verified_workflows_with_env_style.csv  (Stage 2)
# Output:
#   run_metrics_v16_stage3_enhanced.csv
#   run_steps_v16_stage3_breakdown.csv
#
# Uses only GITHUB_TOKEN_1..3 by default
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext"
)

IN_STAGE2_CSV = ROOT_DIR / "run_metrics_v16_from_verified_workflows_with_env_style.csv"
OUT_STAGE3_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
OUT_STEPS_CSV  = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"

MAX_TOKENS_TO_USE = 3
PROCESS_ONLY_RELEVANT_ROWS = True  # set False to recompute for all rows

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# Fetch workflow YAML (recommended). If False, Stage3 behaves closer to your original.
FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000  # limit cache size

# =========================
# Helpers
# =========================
BOM = "\ufeff"

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join_names(names: List[str], max_len: int = 800) -> str:
    s = ",".join(unique_preserve([n.strip() for n in names if n and n.strip()]))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                ck = _clean_key(k)
                clean_row[ck] = (v or "")
            rows.append(clean_row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def ensure_csv(path: Path, fieldnames: List[str]) -> None:
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(path: Path, fieldnames: List[str], row: Dict[str, str]) -> None:
    with path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-metrics-stage3-ttfts/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))

def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

# =========================
# Patterns (names + YAML content)
# =========================
TEST_ANY_RE = re.compile(
    r"(am\s+instrument|connected.*androidtest|androidtest|manageddevice|gmd|devicecheck|"
    r"connectedcheck|espresso|uiautomator|firebase\s+test|test\s+lab|device\s+farm|flank|"
    r"appcenter\s+test|saucectl|browserstack|bstack|maestro\s+cloud|emulator\.wtf)",
    re.IGNORECASE,
)

ENV_ANY_RE = re.compile(
    r"(reactivecircus|android-emulator-runner|emulator\b|avd\b|avdmanager|sdkmanager|"
    r"start[-_ ]emulator|android-wait-for-emulator|adb\s+wait[- ]?for[- ]?device|kvm)",
    re.IGNORECASE,
)

THIRD_PARTY_ANY_RE = re.compile(
    r"(gcloud.*firebase\s+test\s+android\s+run|firebase\s+test\s+lab|flank\s+android\s+run|"
    r"appcenter\s+test\s+run\s+android|saucectl|browserstack|bstack|maestro\s+cloud|"
    r"emulator\.wtf)",
    re.IGNORECASE,
)

ARTIFACT_RE = re.compile(r"(upload[- ]artifact|actions/upload-artifact)", re.IGNORECASE)

def parse_yaml_step_hints(yaml_text: str) -> Dict[str, Dict[str, bool]]:
    """
    Very lightweight heuristic parser:
      builds { step_name_lower: {is_env,is_test,is_3p,is_artifact} }
    Works even without PyYAML.
    """
    hints: Dict[str, Dict[str, bool]] = {}
    if not yaml_text:
        return hints

    lines = yaml_text.splitlines()
    i = 0
    while i < len(lines):
        line = lines[i]
        m = re.match(r"^\s*-\s*name\s*:\s*(.+?)\s*$", line)
        if not m:
            i += 1
            continue
        step_name = m.group(1).strip().strip('"').strip("'")
        key = step_name.lower()
        blob = "\n".join(lines[i : min(len(lines), i + 12)])  # lookahead window
        hints[key] = {
            "is_env": bool(ENV_ANY_RE.search(blob)),
            "is_test": bool(TEST_ANY_RE.search(blob)),
            "is_3p": bool(THIRD_PARTY_ANY_RE.search(blob)),
            "is_artifact": bool(ARTIFACT_RE.search(blob)),
        }
        i += 1
    return hints

# =========================
# Metrics + step breakdown
# =========================
def enhanced_metrics_and_steps(
    jobs: List[Dict],
    run_created_at: str,
    run_started_at: str,
    yaml_hints: Dict[str, Dict[str, bool]],
    styles_text: str,
) -> Tuple[Dict[str, Union[str, int, float, None]], List[Dict[str, str]]]:
    """
    Produces:
      - Stage3 metrics (core/extended + 3P job attribution)
      - Per-step breakdown rows (category + duration)
    Also adds (Item 1):
      - first_test_step_started_at
      - ttfts_seconds
      - ttfts_source
    """
    out: Dict[str, Union[str, int, float, None]] = {
        "queue_seconds": dt_to_seconds(iso_to_dt(run_created_at), iso_to_dt(run_started_at)),

        # legacy metric kept (fallback/compat)
        "time_to_first_instru_seconds": None,

        # keep Stage2 semantics
        "instru_conclusion": "unknown",
        "instru_detect_method": "none",
        "instru_duration_seconds": None,

        "run_duration_seconds": None,
        "runner_labels_union": "",

        "instru_job_count": 0,
        "instru_step_count": 0,
        "instru_job_names": "",
        "instru_step_names": "",
        "instru_total_seconds": None,
        "instru_window_seconds": None,
        "instru_first_started_at": "",
        "instru_last_completed_at": "",
        "instru_share_of_run": None,

        # new in your existing stage3
        "instru_extended_window_seconds": None,
        "env_setup_seconds": None,
        "env_setup_sum_seconds": None,
        "artifact_sum_seconds": None,
        "third_party_job_count": 0,
        "third_party_job_names": "",

        # Item 1 (NEW)
        "first_test_step_started_at": "",
        "ttfts_seconds": None,
        "ttfts_source": "missing",
    }

    step_rows: List[Dict[str, str]] = []
    if not jobs:
        return out, step_rows

    # Run duration + labels
    starts, ends = [], []
    labels_union: Set[str] = set()
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt: starts.append(sdt)
        if edt: ends.append(edt)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                labels_union.add(lab.strip())
    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)
    out["run_duration_seconds"] = run_dur
    out["runner_labels_union"] = ",".join(sorted(labels_union))

    # Boundaries
    core_first: Optional[datetime] = None
    core_last: Optional[datetime] = None
    ext_first: Optional[datetime] = None
    ext_last: Optional[datetime] = None

    first_env_start: Optional[datetime] = None

    # Item 1 anchor (first test-step start across the run)
    first_test_start: Optional[datetime] = None
    first_test_source: str = "missing"  # test_step | third_party_job

    total_test = 0
    total_env = 0
    total_art = 0
    total_any = False

    instru_job_names: List[str] = []
    instru_step_names: List[str] = []
    third_party_job_names: List[str] = []
    third_party_count = 0

    first_match_set = False

    style_third_party = "third-party" in (styles_text or "").lower()

    for j in jobs:
        job_id = str(j.get("id") or "")
        job_name = (j.get("name") or "").strip()
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        job_start = iso_to_dt(j.get("started_at"))
        job_end = iso_to_dt(j.get("completed_at"))
        job_dur = dt_to_seconds(job_start, job_end)

        # 3P detection: job name, step name, OR style hint + any test hint inside steps
        job_is_3p = bool(THIRD_PARTY_ANY_RE.search(job_name)) or any(
            THIRD_PARTY_ANY_RE.search((st.get("name") or "")) for st in steps
        )

        # If YAML hints identify this job as third-party by step name, include it
        if not job_is_3p:
            for st in steps:
                sn = (st.get("name") or "").strip().lower()
                if sn and sn in yaml_hints and yaml_hints[sn].get("is_3p"):
                    job_is_3p = True
                    break

        if style_third_party and not job_is_3p:
            if any(TEST_ANY_RE.search((st.get("name") or "")) for st in steps):
                job_is_3p = True

        if job_is_3p:
            third_party_count += 1
            if job_name:
                third_party_job_names.append(job_name)
                instru_job_names.append(job_name)

            # Attribute full job duration as "test" (core+extended), because the real work is remote
            if job_start and (core_first is None or job_start < core_first): core_first = job_start
            if job_end   and (core_last  is None or job_end   > core_last):  core_last  = job_end
            if job_start and (ext_first  is None or job_start < ext_first):  ext_first  = job_start
            if job_end   and (ext_last   is None or job_end   > ext_last):   ext_last   = job_end

            # Item 1: treat 3P job start as first test start if earlier than current
            if job_start and (first_test_start is None or job_start < first_test_start):
                first_test_start = job_start
                first_test_source = "third_party_job"

            if job_dur is not None:
                total_test += job_dur
                total_any = True
                if not first_match_set:
                    out["instru_conclusion"] = (j.get("conclusion") or "unknown")
                    out["instru_detect_method"] = "third_party_job"
                    out["instru_duration_seconds"] = job_dur
                    first_match_set = True

            # Emit per-step rows (categorized as third_party)
            for st in steps:
                sname = (st.get("name") or "").strip()
                sdt = iso_to_dt(st.get("started_at")) or job_start
                edt = iso_to_dt(st.get("completed_at")) or job_end
                dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
                step_rows.append({
                    "job_id": job_id,
                    "job_name": job_name,
                    "step_name": sname,
                    "category": "third_party",
                    "started_at": sdt.isoformat() if sdt else "",
                    "completed_at": edt.isoformat() if edt else "",
                    "duration_seconds": "" if dur is None else str(dur),
                })

            continue

        # Non-3P path: step classification (names + YAML hints)
        job_is_instruish = bool(TEST_ANY_RE.search(job_name) or ENV_ANY_RE.search(job_name))
        if not job_is_instruish:
            job_is_instruish = any(
                TEST_ANY_RE.search((st.get("name") or "")) or ENV_ANY_RE.search((st.get("name") or ""))
                for st in steps
            )
        if not job_is_instruish:
            for st in steps:
                sn = (st.get("name") or "").strip().lower()
                if sn and sn in yaml_hints and (yaml_hints[sn].get("is_env") or yaml_hints[sn].get("is_test")):
                    job_is_instruish = True
                    break
        if not job_is_instruish:
            continue

        if job_name:
            instru_job_names.append(job_name)

        for st in steps:
            step_name = (st.get("name") or "").strip()
            if not step_name:
                continue

            sn = step_name.lower()
            hint = yaml_hints.get(sn, {})

            is_env = bool(ENV_ANY_RE.search(step_name)) or bool(hint.get("is_env"))
            is_test = bool(TEST_ANY_RE.search(step_name)) or bool(hint.get("is_test"))
            is_art = bool(ARTIFACT_RE.search(step_name)) or bool(hint.get("is_artifact"))

            st_start = iso_to_dt(st.get("started_at")) or job_start
            st_end = iso_to_dt(st.get("completed_at")) or job_end
            st_dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
            if st_dur is None:
                st_dur = dt_to_seconds(job_start, job_end)

            # Category
            if is_art:
                cat = "artifact"
            elif is_test:
                cat = "test"
            elif is_env:
                cat = "env_setup"
            else:
                cat = "other"

            # Emit step row
            step_rows.append({
                "job_id": job_id,
                "job_name": job_name,
                "step_name": step_name,
                "category": cat,
                "started_at": st_start.isoformat() if st_start else "",
                "completed_at": st_end.isoformat() if st_end else "",
                "duration_seconds": "" if st_dur is None else str(st_dur),
            })

            # Extended window includes env+test
            if (is_env or is_test) and st_start and (ext_first is None or st_start < ext_first):
                ext_first = st_start
            if (is_env or is_test) and st_end and (ext_last is None or st_end > ext_last):
                ext_last = st_end

            # Track env first start
            if is_env and st_start and (first_env_start is None or st_start < first_env_start):
                first_env_start = st_start

            # Item 1: Track first test-step start across all test steps
            if is_test and st_start and (first_test_start is None or st_start < first_test_start):
                first_test_start = st_start
                first_test_source = "test_step"

            # Core window (test only)
            if is_test:
                instru_step_names.append(step_name)
                if st_start and (core_first is None or st_start < core_first): core_first = st_start
                if st_end   and (core_last  is None or st_end   > core_last):  core_last  = st_end
                if st_dur is not None:
                    total_test += st_dur
                    total_any = True
                if not first_match_set:
                    out["instru_conclusion"] = (st.get("conclusion") or st.get("status") or "unknown")
                    out["instru_detect_method"] = "step_yaml_assisted"
                    out["instru_duration_seconds"] = st_dur
                    first_match_set = True

            if is_env and st_dur is not None:
                total_env += st_dur
            if is_art and st_dur is not None:
                total_art += st_dur

    out["instru_job_names"] = safe_join_names(instru_job_names)
    out["instru_step_names"] = safe_join_names(instru_step_names)
    out["instru_job_count"] = len(unique_preserve(instru_job_names))
    out["instru_step_count"] = len(unique_preserve(instru_step_names))

    out["third_party_job_count"] = third_party_count
    out["third_party_job_names"] = safe_join_names(third_party_job_names)

    # Core window
    out["instru_first_started_at"] = core_first.isoformat() if core_first else ""
    out["instru_last_completed_at"] = core_last.isoformat() if core_last else ""
    out["instru_window_seconds"] = dt_to_seconds(core_first, core_last)

    # Legacy "time-to-first-instrumentation" (kept)
    if core_first:
        base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)
        out["time_to_first_instru_seconds"] = dt_to_seconds(base_start, core_first)

    # Extended window
    out["instru_extended_window_seconds"] = dt_to_seconds(ext_first, ext_last)

    # env setup delta: (first test start - first env start)
    # Note: uses Item 1's first_test_start (more direct than core_first)
    if first_env_start and first_test_start:
        out["env_setup_seconds"] = dt_to_seconds(first_env_start, first_test_start)

    # Totals
    if total_any:
        out["instru_total_seconds"] = total_test
    out["env_setup_sum_seconds"] = total_env if total_env > 0 else None
    out["artifact_sum_seconds"] = total_art if total_art > 0 else None

    # Share of run (core)
    if out["instru_window_seconds"] is not None and run_dur:
        try:
            out["instru_share_of_run"] = round(float(out["instru_window_seconds"]) / float(run_dur), 6)
        except Exception:
            out["instru_share_of_run"] = None

    # =========================
    # Item 1: TTFTS (primary)
    # =========================
    base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)

    if first_test_start:
        out["first_test_step_started_at"] = first_test_start.isoformat()
        out["ttfts_seconds"] = dt_to_seconds(base_start, first_test_start)
        out["ttfts_source"] = first_test_source
    else:
        # Optional conservative fallback: if we have a core_first window but didn't capture a test step start
        # (rare; mostly for odd/missing step timestamps)
        if core_first:
            out["first_test_step_started_at"] = core_first.isoformat()
            out["ttfts_seconds"] = dt_to_seconds(base_start, core_first)
            out["ttfts_source"] = "fallback_instru_window"
        else:
            out["ttfts_seconds"] = None
            out["ttfts_source"] = "missing"

    return out, step_rows

# =========================
# MAIN
# =========================
def main() -> None:
    # --- overwrite outputs to guarantee one-shot reproducibility ---
    if OUT_STAGE3_CSV.exists():
        OUT_STAGE3_CSV.unlink()
    if OUT_STEPS_CSV.exists():
        OUT_STEPS_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    rows, in_fields = read_csv_rows(IN_STAGE2_CSV)
    if not rows:
        raise RuntimeError("No rows in Stage-2 input CSV.")

    # Relevance filter
    if PROCESS_ONLY_RELEVANT_ROWS:
        def is_relevant(r: Dict[str, str]) -> bool:
            det = (r.get("instru_detect_method", "") or "").strip().lower()
            styles = (r.get("styles", "") or "").lower()
            inv = (r.get("invocation_types", "") or "").lower()
            looks = (r.get("looks_like_instru", "") or "").strip().lower()
            return (
                looks == "yes"
                or det not in ("", "none", "unknown")
                or ("third-party" in styles)
                or ("3p clis" in inv)
            )
        target = [r for r in rows if is_relevant(r)]
    else:
        target = rows

    print(f"Rows total: {len(rows)} | Rows to enhance: {len(target)}")

    # Output fields: keep Stage2 + append new Stage3 fields
    stage3_cols = [
        # Item 1 NEW fields
        "first_test_step_started_at",
        "ttfts_seconds",
        "ttfts_source",

        # legacy + existing stage3 fields
        "instru_detect_method",
        "instru_duration_seconds",
        "instru_total_seconds",
        "instru_window_seconds",
        "instru_extended_window_seconds",
        "env_setup_seconds",
        "env_setup_sum_seconds",
        "artifact_sum_seconds",
        "third_party_job_count",
        "third_party_job_names",
        "stage3_extracted_at_utc",
    ]

    out_fieldnames = list(in_fields)
    for c in stage3_cols:
        if c not in out_fieldnames:
            out_fieldnames.append(c)

    # Steps breakdown file schema
    steps_fields = [
        "full_name",
        "run_id",
        "workflow_identifier",
        "workflow_path",
        "head_sha",
        "styles",
        "job_id",
        "job_name",
        "step_name",
        "category",
        "started_at",
        "completed_at",
        "duration_seconds",
        "stage3_extracted_at_utc",
    ]
    ensure_csv(OUT_STEPS_CSV, steps_fields)

    # Workflow YAML cache (repo, path, sha) -> hints
    yaml_cache: Dict[Tuple[str, str, str], Dict[str, Dict[str, bool]]] = {}

    it = target
    if tqdm is not None:
        it = tqdm(target, desc="Stage3: enhance runs (TTFTS)")

    for r in it:
        full_name = (r.get("full_name") or "").strip()
        run_id = (r.get("run_id") or "").strip()
        created_at = r.get("created_at") or ""
        run_started_at = r.get("run_started_at") or ""
        workflow_identifier = (r.get("workflow_identifier") or "").strip()
        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()
        styles_text = (r.get("styles") or "")

        if not full_name or not run_id:
            continue

        # YAML hints (optional)
        hints: Dict[str, Dict[str, bool]] = {}
        if FETCH_WORKFLOW_YAML and workflow_path and head_sha:
            ck = (full_name, workflow_path, head_sha)
            if ck in yaml_cache:
                hints = yaml_cache[ck]
            else:
                yml = fetch_workflow_yaml(gh, full_name, workflow_path, ref=head_sha)
                hints = parse_yaml_step_hints(yml)
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = hints

        jobs = list_run_jobs(gh, full_name, int(run_id)) or []

        m, step_rows = enhanced_metrics_and_steps(
            jobs=jobs,
            run_created_at=created_at,
            run_started_at=run_started_at,
            yaml_hints=hints,
            styles_text=styles_text,
        )

        # write back stage3 cols
        for k in stage3_cols:
            if k == "stage3_extracted_at_utc":
                r[k] = now_utc_iso()
            else:
                v = m.get(k)
                r[k] = "" if v is None else str(v)

        # append step breakdown rows
        ts = r.get("stage3_extracted_at_utc", now_utc_iso())
        for sr in step_rows:
            append_row(OUT_STEPS_CSV, steps_fields, {
                "full_name": full_name,
                "run_id": run_id,
                "workflow_identifier": workflow_identifier,
                "workflow_path": workflow_path,
                "head_sha": head_sha,
                "styles": styles_text,
                **sr,
                "stage3_extracted_at_utc": ts,
            })

    write_csv(OUT_STAGE3_CSV, out_fieldnames, rows)
    print("Done.")
    print("Stage 3 output:", OUT_STAGE3_CSV)
    print("Step breakdown:", OUT_STEPS_CSV)

if __name__ == "__main__":
    main()


Rows total: 31060 | Rows to enhance: 31060


Stage3: enhance runs (TTFTS): 100%|██████████| 31060/31060 [7:11:02<00:00,  1.20it/s]   


Done.
Stage 3 output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\run_metrics_v16_stage3_enhanced.csv
Step breakdown: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv


In [ ]:
## Stage 4

In [2]:
# ============================================================
# Stage 4 (NEW for Item 3: Workload/Test Signature Layer)
#
# Reads Stage 3 outputs and emits run-level workload signatures.
#
# Inputs:
#   - run_metrics_v16_stage3_enhanced.csv
#   - run_steps_v16_stage3_breakdown.csv
#
# Outputs (suggested):
#   - run_workload_signature_v1.csv   (1 row per run)
#
# Strategy (ranked):
#   1) Prefer Artifacts: list artifacts for the run, fingerprint artifact names.
#      Optional: download + parse small junit xml to count testcases/suites.
#   2) Fallback to workflow YAML + step inference: extract gradle tasks / 3P CLI hints
#   3) (Optional later) logs inference
#
# Notes:
#   - Pinned to head_sha (already in Stage 3 inputs)
#   - Designed to be independent from Stage 3
# ============================================================

import base64
import csv
import hashlib
import random
import re
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests
import xml.etree.ElementTree as ET

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext"
)

IN_STAGE3_CSV   = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
IN_STEPS_CSV    = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"
OUT_SIG_CSV     = ROOT_DIR / "run_workload_signature_v1.csv"

MAX_TOKENS_TO_USE = 3

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 90
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# If True, will download small artifacts and attempt junit XML parsing.
DOWNLOAD_AND_PARSE_ARTIFACTS = True  # enabled for workload evidence v2
MAX_ARTIFACT_ZIP_BYTES = 25 * 1024 * 1024  # 25MB safety cap

# Fetch workflow YAML to extract task/CLI signatures (recommended)
FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000

# =========================
# Helpers
# =========================
BOM = "\ufeff"

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                clean_row[_clean_key(k)] = (v or "")
            rows.append(clean_row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def safe_lower(s: str) -> str:
    return (s or "").strip().lower()

def uniq_sorted(items: Iterable[str]) -> List[str]:
    s = {x.strip() for x in items if x and x.strip()}
    return sorted(s)

def stable_hash(parts: List[str]) -> str:
    """
    Stable hash over normalized signature components.
    """
    blob = "\n".join([p.strip() for p in parts if p and p.strip()])
    return hashlib.sha1(blob.encode("utf-8", errors="ignore")).hexdigest()

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

# =========================
# GitHub API client (same style as Stage 3)
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-metrics-stage4-signature/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def list_run_artifacts(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/artifacts"
    return list(gh.paginate(url, params={}, item_key="artifacts"))

def download_artifact_zip(gh: GitHubClient, full_name: str, artifact_id: int) -> Optional[bytes]:
    """
    Downloads artifact as a zip. This can be heavy; guarded by caps.
    """
    url = f"https://api.github.com/repos/{full_name}/actions/artifacts/{artifact_id}/zip"
    idx = gh._pick_idx()
    st = gh.tokens[idx]
    gh.session.headers["Authorization"] = f"Bearer {st.token}"
    try:
        r = gh.session.get(url, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S), stream=True)
        if r.status_code != 200:
            return None
        data = r.content
        if data and len(data) <= MAX_ARTIFACT_ZIP_BYTES:
            return data
        return None
    except requests.exceptions.RequestException:
        return None

def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

# =========================
# Signature inference patterns
# =========================
GRADLE_TASK_RE = re.compile(
    r"(?:\./gradlew|\bgradle[w]?\b)\s+([^\n\r#]+)",
    re.IGNORECASE
)

# Rough extraction of gradle tasks from command tail:
# e.g., "./gradlew :app:connectedDebugAndroidTest --stacktrace" -> tasks = [":app:connectedDebugAndroidTest"]
GRADLE_TASK_TOKEN_RE = re.compile(r"(?:(?::[\w\-.]+)+|[\w\-.]+)", re.IGNORECASE)

THIRD_PARTY_PROVIDER_RE = re.compile(
    r"(firebase\s+test\s+lab|gcloud\s+firebase\s+test|flank|appcenter|browserstack|bstack|saucectl|maestro\s+cloud|emulator\.wtf)",
    re.IGNORECASE
)

def extract_gradle_tasks_from_yaml(yaml_text: str) -> List[str]:
    """
    Extracts tasks from lines containing gradlew/gradle invocations.
    Heuristic: pull tokens until flags and filter out obvious flags.
    """
    tasks: List[str] = []
    if not yaml_text:
        return tasks
    for m in GRADLE_TASK_RE.finditer(yaml_text):
        tail = (m.group(1) or "").strip()
        # Split on typical shell separators, keep first segment
        tail = tail.split("&&")[0].split(";")[0].strip()
        # Tokenize; stop when flags start (--) but allow tasks before
        toks = tail.split()
        for t in toks:
            if t.startswith("-"):
                continue
            # exclude common non-task words
            if t.lower() in ("cd", "echo", "export", "set"):
                continue
            if GRADLE_TASK_TOKEN_RE.fullmatch(t):
                tasks.append(t)
    # Normalize: keep only likely tasks (contain connected/device/check/test)
    keep = []
    for t in tasks:
        tl = t.lower()
        if any(k in tl for k in ("connected", "androidtest", "device", "check", "test", "manageddevice", "gmd")):
            keep.append(t)
    return uniq_sorted(keep)

def infer_test_driver(styles: str, steps_blob: str, yaml_text: str) -> str:
    s = safe_lower(styles)
    b = safe_lower(steps_blob)
    y = safe_lower(yaml_text)

    # third-party providers
    if "third-party" in s:
        m = THIRD_PARTY_PROVIDER_RE.search(styles) or THIRD_PARTY_PROVIDER_RE.search(steps_blob) or THIRD_PARTY_PROVIDER_RE.search(yaml_text)
        if m:
            hit = safe_lower(m.group(1))
            if "firebase" in hit or "test lab" in hit or "gcloud" in hit:
                return "firebase_test_lab"
            if "flank" in hit:
                return "flank"
            if "appcenter" in hit:
                return "appcenter"
            if "browserstack" in hit or "bstack" in hit:
                return "browserstack"
            if "saucectl" in hit:
                return "sauce"
            if "maestro" in hit:
                return "maestro_cloud"
            if "emulator.wtf" in hit:
                return "emulator_wtf"
        return "third_party_unknown"

    # local/gradle
    if ("gradle" in b) or ("gradlew" in y) or ("connected" in b) or ("androidtest" in b):
        return "gradle"

    return "unknown"

def junit_counts_from_zip_bytes(zip_bytes: bytes) -> Tuple[int, int]:
    """
    Returns (suite_count, testcase_count) by scanning .xml files that look like JUnit.
    Best-effort, not perfect.
    """
    suite_count = 0
    case_count = 0
    try:
        with zipfile.ZipFile(BytesIO(zip_bytes)) as z:
            for name in z.namelist():
                nl = name.lower()
                if not nl.endswith(".xml"):
                    continue
                if ("junit" not in nl) and ("test-" not in nl) and ("androidtest" not in nl) and ("connected" not in nl):
                    continue
                try:
                    data = z.read(name)
                    # parse XML lightly
                    root = ET.fromstring(data)
                    # Common JUnit roots: testsuite / testsuites
                    if root.tag.lower().endswith("testsuite"):
                        suite_count += 1
                        case_count += len(root.findall(".//testcase"))
                    elif root.tag.lower().endswith("testsuites"):
                        suites = root.findall(".//testsuite")
                        suite_count += len(suites)
                        case_count += len(root.findall(".//testcase"))
                except Exception:
                    continue
    except Exception:
        return (0, 0)
    return (suite_count, case_count)


def junit_metrics_from_zip_bytes(zip_bytes: bytes, max_testcases_for_fingerprint: int = 5000) -> Tuple[int, int, int, int, int, str]:
    """
    Returns (suite_count, testcase_count, failed_count, error_count, skipped_count, executed_tests_fingerprint).
    Fingerprint is a SHA1 over sorted testcase identifiers (classname::name) truncated to a stable hex string.
    Best-effort: if no testcase identifiers found, fingerprint is ''.
    """
    suite_count = 0
    case_count = 0
    failed = 0
    errors = 0
    skipped = 0
    ids: List[str] = []
    try:
        with zipfile.ZipFile(BytesIO(zip_bytes)) as z:
            for name in z.namelist():
                nl = name.lower()
                if not nl.endswith(".xml"):
                    continue
                # Heuristics to avoid parsing unrelated XMLs
                if ("junit" not in nl) and ("test-" not in nl) and ("test_results" not in nl) and ("androidtest" not in nl) and ("connected" not in nl) and ("surefire" not in nl):
                    continue
                try:
                    data = z.read(name)
                    root = ET.fromstring(data)
                except Exception:
                    continue

                # accumulate suites and testcases
                if root.tag.lower().endswith("testsuite"):
                    suite_count += 1
                elif root.tag.lower().endswith("testsuites"):
                    suites = root.findall(".//testsuite")
                    suite_count += len(suites)

                # read counters from testsuite attributes when present (more reliable than counting testcase nodes only)
                for ts in root.findall(".//testsuite") if root.tag.lower().endswith("testsuites") else ([root] if root.tag.lower().endswith("testsuite") else []):
                    try:
                        case_count += int(ts.get("tests") or 0)
                    except Exception:
                        pass
                    try:
                        failed += int(ts.get("failures") or 0)
                    except Exception:
                        pass
                    try:
                        errors += int(ts.get("errors") or 0)
                    except Exception:
                        pass
                    try:
                        skipped += int(ts.get("skipped") or 0)
                    except Exception:
                        pass

                # collect testcase identifiers for fingerprint (bounded)
                if len(ids) < max_testcases_for_fingerprint:
                    for tc in root.findall(".//testcase"):
                        cls = tc.get("classname") or ""
                        nm = tc.get("name") or ""
                        if cls or nm:
                            ids.append(f"{cls}::{nm}")
                        if len(ids) >= max_testcases_for_fingerprint:
                            break
    except Exception:
        return (0, 0, 0, 0, 0, "")

    # Fallback testcase counting if suite attributes were absent
    if case_count == 0:
        try:
            case_count = len(ids)
        except Exception:
            case_count = 0

    fp = ""
    if ids:
        ids_sorted = sorted(set(ids))
        fp = hashlib.sha1("\n".join(ids_sorted).encode("utf-8", errors="ignore")).hexdigest()

    return (suite_count, case_count, failed, errors, skipped, fp)

# =========================
# MAIN
# =========================
def main() -> None:
    # --- overwrite outputs to guarantee one-shot reproducibility ---
    if OUT_SIG_CSV.exists():
        OUT_SIG_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    runs, _ = read_csv_rows(IN_STAGE3_CSV)
    steps, _ = read_csv_rows(IN_STEPS_CSV)

    if not runs:
        raise RuntimeError("Stage 3 enhanced CSV is empty.")
    if not steps:
        print("[warn] Steps breakdown CSV is empty; signatures will rely on artifacts + styles + YAML only.")

    # Index step rows by (full_name, run_id)
    steps_by_run: Dict[Tuple[str, str], List[Dict[str, str]]] = {}
    for s in steps:
        key = ((s.get("full_name") or "").strip(), (s.get("run_id") or "").strip())
        if not key[0] or not key[1]:
            continue
        steps_by_run.setdefault(key, []).append(s)

    # YAML cache (repo, path, sha) -> yaml text
    yaml_cache: Dict[Tuple[str, str, str], str] = {}

    out_rows: List[Dict[str, str]] = []

    it = runs
    if tqdm is not None:
        it = tqdm(runs, desc="Stage4: workload signatures")

    for r in it:
        full_name = (r.get("full_name") or "").strip()
        run_id_s = (r.get("run_id") or "").strip()
        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()
        styles = (r.get("styles") or "")

        if not full_name or not run_id_s:
            continue

        run_key = (full_name, run_id_s)
        sr_list = steps_by_run.get(run_key, [])

        # Build a step blob for inference
        test_step_names = [s.get("step_name", "") for s in sr_list if safe_lower(s.get("category", "")) in ("test", "third_party")]
        test_job_names  = [s.get("job_name", "") for s in sr_list if safe_lower(s.get("category", "")) in ("test", "third_party")]
        steps_blob = "\n".join(test_job_names + test_step_names)

        # 1) Artifacts
        artifacts = list_run_artifacts(gh, full_name, int(run_id_s)) or []
        artifact_names = uniq_sorted([a.get("name", "") for a in artifacts if a.get("name")])
        artifact_name_fingerprint = "|".join(artifact_names)

        
junit_suites = 0
        junit_cases = 0
        junit_failed = 0
        junit_errors = 0
        junit_skipped = 0
        executed_tests_fingerprint = ""
        results_artifact_present = False
        results_artifact_types: Set[str] = set()

        # Heuristic: artifact names indicate presence of results bundles
        for an in artifact_names:
            anl = (an or "").lower()
            if any(k in anl for k in ["test-result", "test_results", "junit", "androidtest", "instrumentation", "connected", "report", "reports", "results"]):
                results_artifact_present = True
                if "junit" in anl or "test-" in anl:
                    results_artifact_types.add("junit_xml_or_bundle")
                elif "report" in anl or "reports" in anl:
                    results_artifact_types.add("report_bundle")
                else:
                    results_artifact_types.add("results_bundle")

        if DOWNLOAD_AND_PARSE_ARTIFACTS and artifacts:
            # parse a small subset to keep runtime manageable
            # prioritize artifacts likely to contain results
            cand = []
            for a in artifacts:
                n = (a.get("name") or "").lower()
                score = 0
                for kw in ["test-results", "test_results", "junit", "androidtest", "instrumentation", "connected", "report", "reports", "results"]:
                    if kw in n:
                        score += 1
                cand.append((score, a))
            cand.sort(key=lambda t: t[0], reverse=True)
            for score, a in cand[:5]:
                if score == 0:
                    continue
                try:
                    aid = int(a.get("id"))
                except Exception:
                    continue
                zip_bytes = download_artifact_zip(gh, full_name, aid)
                if not zip_bytes:
                    continue
                sc, cc, ff, ee, ss, fp = junit_metrics_from_zip_bytes(zip_bytes)
                junit_suites += sc
                junit_cases += cc
                junit_failed += ff
                junit_errors += ee
                junit_skipped += ss
                if fp and not executed_tests_fingerprint:
                    executed_tests_fingerprint = fp
                results_artifact_present = True
                results_artifact_types.add("parsed_zip")

        # Determine evidence level for reviewer-friendly validity
        if executed_tests_fingerprint or (junit_cases > 0) or (junit_suites > 0):
            workload_evidence_level = "A"  # executed results evidence
        elif artifact_names:
            workload_evidence_level = "B"  # artifact presence only
        else:
            workload_evidence_level = "C"  # invocation inference or unknown

        results_artifact_types_str = "|".join(sorted(results_artifact_types)) if results_artifact_types else ""

        # 2) YAML extraction

        yaml_text = ""
        gradle_tasks = []
        if FETCH_WORKFLOW_YAML and workflow_path and head_sha:
            ck = (full_name, workflow_path, head_sha)
            if ck in yaml_cache:
                yaml_text = yaml_cache[ck]
            else:
                yaml_text = fetch_workflow_yaml(gh, full_name, workflow_path, ref=head_sha)
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = yaml_text
            gradle_tasks = extract_gradle_tasks_from_yaml(yaml_text)

        # Infer test driver/provider
        test_driver = infer_test_driver(styles, steps_blob, yaml_text)

        # Determine signature kind preference:
        # artifact if any artifacts exist, else yaml if tasks found or provider detected, else steps, else unknown
        signature_kind = "unknown"
        if artifact_names:
            signature_kind = "artifact"
        elif gradle_tasks or THIRD_PARTY_PROVIDER_RE.search(styles) or THIRD_PARTY_PROVIDER_RE.search(steps_blob) or THIRD_PARTY_PROVIDER_RE.search(yaml_text):
            signature_kind = "yaml"
        elif test_step_names:
            signature_kind = "steps"

        # Normalize modules from gradle tasks (e.g., :app:connectedDebugAndroidTest -> module=:app)
        modules: Set[str] = set()
        for t in gradle_tasks:
            if t.startswith(":"):
                parts = t.split(":")
                if len(parts) >= 2 and parts[1]:
                    modules.add(":" + parts[1])

        modules_sorted = sorted(modules)
        gradle_task_fingerprint = "|".join(gradle_tasks)
        module_fingerprint = "|".join(modules_sorted)

        # Provider string (if any)
        provider = ""
        mprov = THIRD_PARTY_PROVIDER_RE.search(styles) or THIRD_PARTY_PROVIDER_RE.search(steps_blob) or THIRD_PARTY_PROVIDER_RE.search(yaml_text)
        if mprov:
            provider = safe_lower(mprov.group(1))

        # Signature hash over stable components
        sig_parts = [
            f"kind={signature_kind}",
            f"driver={test_driver}",
            f"provider={provider}",
            f"artifact_names={artifact_name_fingerprint}",
            f"gradle_tasks={gradle_task_fingerprint}",
            f"modules={module_fingerprint}",
            f"head_sha={head_sha}",
        ]
        signature_hash = stable_hash(sig_parts)

        out_rows.append({
            "full_name": full_name,
            "run_id": run_id_s,
            "workflow_path": workflow_path,
            "head_sha": head_sha,
            "styles": styles,
            "signature_kind": signature_kind,
            "test_driver": test_driver,
            "provider": provider,
            "artifact_count": str(len(artifacts)),
            "artifact_name_fingerprint": artifact_name_fingerprint,
            "gradle_task_fingerprint": gradle_task_fingerprint,
            "module_fingerprint": module_fingerprint,
            "junit_suite_count": str(junit_suites),
            "junit_testcase_count": str(junit_cases),
            "junit_failed_count": str(junit_failed),
            "junit_error_count": str(junit_errors),
            "junit_skipped_count": str(junit_skipped),
            "executed_tests_fingerprint": executed_tests_fingerprint,
            "results_artifact_present": str(bool(results_artifact_present)),
            "results_artifact_types": results_artifact_types_str,
            "workload_evidence_level": workload_evidence_level,
            "signature_hash": signature_hash,
            "stage4_extracted_at_utc": now_utc_iso(),
        })

    out_fields = [
        "full_name",
        "run_id",
        "workflow_path",
        "head_sha",
        "styles",
        "signature_kind",
        "test_driver",
        "provider",
        "artifact_count",
        "artifact_name_fingerprint",
        "gradle_task_fingerprint",
        "module_fingerprint",
        "junit_suite_count",
        "junit_testcase_count",
        "junit_failed_count",
        "junit_error_count",
        "junit_skipped_count",
        "executed_tests_fingerprint",
        "results_artifact_present",
        "results_artifact_types",
        "workload_evidence_level",
        "signature_hash",
        "stage4_extracted_at_utc",
    ]

    write_csv(OUT_SIG_CSV, out_fields, out_rows)
    print("Done.")
    print("Stage 4 output:", OUT_SIG_CSV)

if __name__ == "__main__":
    main()


Stage4: workload signatures:  28%|██▊       | 8678/31060 [51:44<1:51:07,  3.36it/s]

[rate-limit] sleeping 801s until reset...


Stage4: workload signatures:  36%|███▌      | 11177/31060 [1:20:43<1:37:51,  3.39it/s]   

[rate-limit] sleeping 4s until reset...


Stage4: workload signatures:  52%|█████▏    | 16127/31060 [1:51:46<1:55:38,  2.15it/s]

...

[rate-limit] sleeping 1171s until reset...


Stage4: workload signatures:  84%|████████▍ | 26109/31060 [3:17:18<29:07,  2.83it/s]     

[rate-limit] sleeping 212s until reset...


Stage4: workload signatures: 100%|██████████| 31060/31060 [3:47:59<00:00,  2.27it/s]   


Done.
Stage 4 output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026_Ext\run_workload_signature_v1.csv
